# INDEX

[**INDEX**](#index)

[**IMPORTS**](#imports)

[**GAME LOGIC**](#game-logic)  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**GameState**](#class-gamestate)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[**Play Human v Human**](#testing-game)</span>

[**MCTS**](#mcts)  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**Baseline MCTS**](#baseline-mcts)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[**Play against Baseline MCTS**](#testing-baseline-mcts)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**Stupid MCTS**](#stupid-mcts)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[**Play against sMCTS**](#testing-stupid-mcts)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**Dynamic Variance-Scaled cPUCT**](#dynamic-variance-scaled-cpuct)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**UCB-V (Variance-Aware UCB)**](#ucb-v-variance-aware-ucb)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**Global Std (Scale-Independent UCT)**](#global-std-scale-independent-uct)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**Heuristic Rollout MCTS**](#heuristic-rollout-mcts)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**RAVE**](#rave-rapid-action-value-estimation)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**Progressive Widening + Early Termination**](#progressive-widening--early-termination)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**Forking (Scenario Diversification)**](#forking-scenario-diversification)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**Root Policy Softmax Temperature**](#root-policy-softmax-temperature)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**ELO Rating System & Tournament**](#elo-rating-system--tournament)</span>

[**DECISION TREE**](#decision-tree)  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**DT Model**](#dt-model)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[**Test with Iris**](#testing-dt-with-iris)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;[**Creating a Dataset**](#creating-pop-out-dataset)</span>  
<span style="font-size:0.88em">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[**Play against DT**](#testing-dt-for-pop-out)</span>

---

In [ ]:
from collections import deque
from copy import deepcopy
from IPython.display import clear_output
import math
import random
import pandas as pd
import numpy as np
import csv

# GAME LOGIC

## class GameState
### Variables:
```py
    h: int
    w: int
    board: list[deque]
    player: int
    prevStates: dict
    winner: (int | None)
    gameOver: bool
```
### Methods:
```py
    isFull(self: Self@GameState) -> bool
    getWinner(self: Self@GameState) -> (int | None)
    checkRepetition(self: Self@GameState) -> bool
    getValid(self: Self@GameState) -> list
    move(
        self: Self@GameState,
        col: int,
        t: str
    ) -> bool
    printB(self: Self@GameState) -> None
```


In [74]:
class GameState:
    def __init__(self):
        self.h=6
        self.w=7
        self.board=[deque(maxlen=self.h) for _ in range(self.w)]
        self.player=1 # X=1; O=2
        self.prevStates = {}
        self.winner=None
        self.gameOver=False

    def isFull(self):
        """Check if Board is full"""
        return all(len(col) == self.h for col in self.board)

    def getWinner(self):
        """Checks for winner, returns None if no winner exists"""
    
        def check_direction(x, y, dx, dy):
            start = self.board[x][y]
            if start == 0:
                return None
            for i in range(1, 4):
                nx = x + dx * i
                ny = y + dy * i

                if nx < 0 or nx >= self.w or ny < 0 or ny >= len(self.board[nx]):
                    return None

                if self.board[nx][ny] != start:
                    return None

            return start

        winners = set()

        for x in range(self.w):
            for y in range(len(self.board[x])):

                for dx, dy in [(1,0), (0,1), (1,1), (1,-1)]:
                    result = check_direction(x, y, dx, dy)
                    if result is not None:
                        winners.add(result)

        if len(winners) == 1:
            return winners.pop()
        elif len(winners) > 1:
            return self.player
        else:
            return None

    def checkRepetition(self):
        """Check if the position has been reached 3 times"""
        s = tuple(tuple(col) for col in self.board) # Tuples are comparable
        if s not in self.prevStates:
            self.prevStates[s]=0
        self.prevStates[s]+=1

        return self.prevStates[s] >= 3

    def getValid(self):
        """Returns list of valid moves"""

        out=[]
        if self.gameOver:
            return out
        for c in range(self.w):
            col=self.board[c]
            if len(col) > 0 and col[0] == self.player:
                out.append((c, 'p'))
            if len(col) < self.h:
                out.append((c, 'i'))
        return out

    def move(self,col,t): # t = 'i' "insert" | 'p' "pop"
        """Handles move of type t in column col"""

        lm = self.getValid()
        if len(lm) == 0:
            self.gameOver=True
            return False
        if (col, t) not in lm:
            # Draw on insert to full board to avoid getting stuck on HvH games,
            # this part should be unreachable now
            if t == 'i' and 0 <= col < self.w and self.isFull():
                self.gameOver=True
            return False

        if t == 'i':
            self.board[col].append(self.player)
        else:
            self.board[col].popleft()

        winner = self.getWinner()
        if winner:
            self.gameOver=True
            self.winner=winner
        if self.isFull():
            self.gameOver=True
        if self.checkRepetition():
            self.gameOver=True

        self.player=3-self.player
        return True

    def printB(self):
        """Print the board (ASCII) to standard output"""
        print("+-------+")

        for row in range(self.h - 1, -1, -1):
            print("|", end="")
            for col in range(self.w):
                if row >= len(self.board[col]):
                    print(" ", end="")
                elif self.board[col][row] == 1:
                    print("X", end="")
                elif self.board[col][row] == 2:
                    print("O", end="")
                else:
                    print(" ", end="")
            print("|")

        print("+-------+")
        print("|0123456|")
        print("+-------+")

        return None

### Testing game: 
Run to play the game Human vs Human.  
The input is done on separate lines with a single character `p` (for pop) or `i` (for insert) in the first line followed by an integer representing the column number to play on in the second line.

In [3]:
g=GameState()

g.printB()
while not g.gameOver:
    t=input("Type: ") # 'i' <- insert | 'p' <- pop
    c=int(input("Column: "))

    if not g.move(c,t):
        print("Invalid Move")
    else:
        clear_output()
        g.printB()

print("--- Winner: ",g.winner,"---")

+-------+
|       |
|       |
|       |
|      O|
|    OOX|
|  XXXXO|
+-------+
|0123456|
+-------+
--- Winner:  1 ---


# MCTS

## Baseline MCTS

In [75]:
class MCTSNode:
    def __init__(self, state: GameState, parent=None, action=None):
        self.state=state
        self.parent= parent
        self.action=action
        if parent is None:
            self.player=state.player
        else:
            self.player=3-state.player
        self.children=[]
        self.visits=0
        self.wins=0.0
        self.untriedActions=state.getValid()

    def isTerminal(self):
        """Check if node is terminal"""
        return self.state.gameOver
    
    def fullyExpanded(self):
        """Check if all actions are explored"""
        return len(self.untriedActions) == 0
    
    def expand(self):
        """Expand node"""
        (c,t) = self.untriedActions.pop()
        newState = deepcopy(self.state)
        newState.move(c,t)

        child = MCTSNode(newState,parent=self,action=(c,t))
        self.children.append(child)
        return child

    def selectChild(self,c=1.4):
        for child in self.children:
            if child.visits == 0:
                return child
        
        def ucb(child: MCTSNode):
            exploit = child.wins / child.visits
            explore = c * math.sqrt(math.log(self.visits) / child.visits)
            return exploit + explore
    
        return max (self.children, key=ucb)

    def rollout(self):
        """Plays random moves until the game finishes"""
        state = deepcopy(self.state)
        while True:
            if state.gameOver:
                return state.winner
            (c,t) = random.choice(state.getValid())
            state.move(c,t)

    def backpropagate(self,winner):
        """Updates the wins and visits from simulation results"""
        self.visits+=1

        if winner is None:
            self.wins += 0.5
        elif winner == self.player:
            self.wins += 1.0

        if self.parent:
            self.parent.backpropagate(winner)

        return

# -- end of class --

def mctsSearch(rootState: GameState, iterations = 500):
    root = MCTSNode(rootState)
    for _ in range (iterations):
        node = root
        while not node.isTerminal() and node.fullyExpanded():
            node = node.selectChild()
        if not node.isTerminal() and not node.fullyExpanded():
            node=node.expand()
        winner = node.rollout()
        node.backpropagate(winner)

    best = max(root.children,key=lambda c: c.visits)
    return best.action

### Testing baseline MCTS
Run to play the game Human vs MCTS.  
Change the line `if g.player == 1:` into `if g.player == 2:` to go second.  

The input is done on separate lines with a single character `p` (for pop) or `i` (for insert) in the first line followed by an integer representing the column number to play on in the second line.

In [8]:
g=GameState()

g.printB()
while not g.gameOver:
    if g.player == 1: # Chage to 2 to let the algorithm start first
        t=input("Type: ") # 'i' <- insert | 'p' <- pop
        c=int(input("Column: "))

        if not g.move(c,t):
            print("Invalid Move")
        else:
            clear_output()
            g.printB()
    else:
        (c,t) = mctsSearch(g,iterations=1000)

        g.move(c,t)
        clear_output()
        g.printB()

print("--- Winner:",g.winner,"---")

+-------+
|     O |
|     O |
|    XO |
|    OO |
|   XXX |
|   XOX |
+-------+
|0123456|
+-------+
--- Winner: 2 ---


## Stupid MCTS
Easier difficulty implemented by rolling a random move one-quarter of the time instead of trying to find the best strategy

In [76]:
def mctstupidSearch(rootState: GameState, iterations = 500):
    if random.random() < 0.25:
        moves = rootState.getValid()
        #print("Oops!") # See how often it misses during training
        return random.choice(moves)
    return mctsSearch(rootState,iterations=iterations)

### Testing Stupid MCTS

#### Human vs sMCTS
Run to play the game Human vs sMCTS.  
Change the line `if g.player == 1:` into `if g.player == 2:` to go second.  

The input is done on separate lines with a single character `p` (for pop) or `i` (for insert) in the first line followed by an integer representing the column number to play on in the second line.

In [10]:
g=GameState()

g.printB()
while not g.gameOver:
    if g.player == 1: # Chage to 2 to let the algorithm start first
        t=input("Type: ") # 'i' <- insert | 'p' <- pop
        c=int(input("Column: "))

        if not g.move(c,t):
            print("Invalid Move")
        else:
            clear_output()
            g.printB()
    else:
        (c,t) = mctstupidSearch(g,iterations=1000)

        g.move(c,t)
        clear_output()
        g.printB()

print("--- Winner:",g.winner,"---")

+-------+
|       |
|       |
|  XO O |
|  XXOO |
|  XOXX |
|  OXOXX|
+-------+
|0123456|
+-------+
--- Winner: 2 ---


#### sMCTS vs MCTS
Run to play 10 games of sMCTS vs baseline MCTS.  
Change the line `if g.player == 1:` into `if g.player == 2:` to switch their order of play.  

In [11]:
for _ in range (10):
    g=GameState()

    #g.printB()
    while not g.gameOver:
        if g.player == 1:
            (c,t) = mctsSearch(g,iterations=1000)
            g.move(c,t)
        else:
            (c,t) = mctstupidSearch(g,iterations=1000)
            g.move(c,t)

    print("--- Winner:",g.winner,"---")

--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---


## Dynamic Variance-Scaled cPUCT

Instead of a fixed constant $C$ in the UCT formula, we scale $C$ proportionally to the standard deviation of playout utilities at that node:

$$UCT_{dyn}(n) = \frac{Q(n)}{N(n)} + C_{base} \cdot \sigma(n) \cdot \sqrt{\frac{\ln N(\text{parent})}{N(n)}}$$

Here $\sigma(n)$ is the empirical standard deviation of playout outcomes. High-variance nodes (uncertain tactical positions) receive stronger exploration; stable nodes favour exploitation.

**Estimating $\sigma(n)$.** Each rollout that reaches the node contributes a scalar reward $r \in \{0,\, 0.5,\, 1\}$ during backpropagation: $1$ if the side-to-move at that node wins the simulated game, $0$ if it loses, and $0.5$ on draws. Over $N(n)$ visits we keep the usual total reward $Q(n) = \sum r$ (implemented as `wins`) and also $\sum r^2$ (`sum_sq`). The empirical variance of rewards is

$$\mathrm{Var}(n) = \frac{\sum r^2}{N(n)} - \left(\frac{Q(n)}{N(n)}\right)^2 = \mathbb{E}[r^2] - \mathbb{E}[r]^2\,.$$

We clamp $\mathrm{Var}(n)$ below at $0$ (numerical noise), add a tiny jitter $\varepsilon$ (e.g. $0.01$), and set $\sigma(n) = \sqrt{\max(\mathrm{Var}(n),\, 0) + \varepsilon}$. Thus $\sigma$ grows when rollouts disagree (mixed wins/losses) and shrinks when outcomes are almost always the same.



In [77]:
class MCTSNodeDynamic(MCTSNode):
    """MCTSNode that tracks variance of playout results for dynamic cPUCT."""
    def __init__(self, state: GameState, parent=None, action=None):
        super().__init__(state, parent, action)
        self.sum_sq = 0.0

    def selectChild(self, c=1.4):
        for child in self.children:
            if child.visits == 0:
                return child

        def ucb_dynamic(child):
            mean = child.wins / child.visits
            variance = max(child.sum_sq / child.visits - mean * mean, 0.0)
            stdev = math.sqrt(variance + 0.01)
            c_dyn = c * stdev
            exploit = mean
            explore = c_dyn * math.sqrt(math.log(self.visits) / child.visits)
            return exploit + explore

        return max(self.children, key=ucb_dynamic)

    def expand(self):
        (col, t) = self.untriedActions.pop()
        newState = deepcopy(self.state)
        newState.move(col, t)
        child = MCTSNodeDynamic(newState, parent=self, action=(col, t))
        self.children.append(child)
        return child

    def backpropagate(self, winner):
        self.visits += 1
        if winner is None:
            reward = 0.5
        elif winner == self.player:
            reward = 1.0
        else:
            reward = 0.0
        self.wins += reward
        self.sum_sq += reward * reward
        if self.parent:
            self.parent.backpropagate(winner)


def mctsDynamicSearch(rootState: GameState, iterations=500):
    root = MCTSNodeDynamic(rootState)
    for _ in range(iterations):
        node = root
        while not node.isTerminal() and node.fullyExpanded():
            node = node.selectChild()
        if not node.isTerminal() and not node.fullyExpanded():
            node = node.expand()
        winner = node.rollout()
        node.backpropagate(winner)
    best = max(root.children, key=lambda c: c.visits)
    return best.action

## UCB-V (Variance-Aware UCB)

Following Audibert et al. (2009), we use Bernstein's inequality to incorporate the empirical variance of rewards. The formula contains **two exploration terms**:

$$\text{UCB-V}(i) = \bar{X}_i + \sqrt{\frac{2\,\hat\sigma_i^2 \ln N}{n_i}} + \frac{3\ln N}{n_i}$$

The first term scales with the variance—strong exploration at uncertain nodes, weak exploration once a node looks solved. The second term ($3\ln N / n_i$) is an **additive floor** that enforces a minimum exploration rate independent of the variance, fixing the issue that the original Dynamic variant could collapse to zero when $\sigma^2 = 0$.


In [78]:
class MCTSNodeUCBV(MCTSNode):
    """MCTSNode using UCB-V (Audibert et al., 2009) for child selection."""
    def __init__(self, state: GameState, parent=None, action=None):
        super().__init__(state, parent, action)
        self.sum_sq = 0.0

    def selectChild(self, c=None):
        for child in self.children:
            if child.visits == 0:
                return child

        logN = math.log(self.visits)

        def ucb_v(child):
            mean = child.wins / child.visits
            variance = max(child.sum_sq / child.visits - mean * mean, 0.0)
            explore_var = math.sqrt(2 * variance * logN / child.visits)
            explore_bonus = 3 * logN / child.visits
            return mean + explore_var + explore_bonus

        return max(self.children, key=ucb_v)

    def expand(self):
        (col, t) = self.untriedActions.pop()
        newState = deepcopy(self.state)
        newState.move(col, t)
        child = MCTSNodeUCBV(newState, parent=self, action=(col, t))
        self.children.append(child)
        return child

    def backpropagate(self, winner):
        self.visits += 1
        if winner is None:
            reward = 0.5
        elif winner == self.player:
            reward = 1.0
        else:
            reward = 0.0
        self.wins += reward
        self.sum_sq += reward * reward
        if self.parent:
            self.parent.backpropagate(winner)


def mctsUCBVSearch(rootState: GameState, iterations=500):
    root = MCTSNodeUCBV(rootState)
    for _ in range(iterations):
        node = root
        while not node.isTerminal() and node.fullyExpanded():
            node = node.selectChild()
        if not node.isTerminal() and not node.fullyExpanded():
            node = node.expand()
        winner = node.rollout()
        node.backpropagate(winner)
    best = max(root.children, key=lambda c: c.visits)
    return best.action

## Global Std (Scale-Independent UCT)

Following Schmocker et al. (2024), we replace the fixed exploration constant $C$ with $C \cdot \sigma_{\text{global}}$, where $\sigma_{\text{global}}$ is the standard deviation of Q-values ($Q = \text{wins}/\text{visits}$) taken over **all** sufficiently visited nodes in the tree:

$$\text{UCT}_{\text{GStd}}(i) = \bar{X}_i + C \cdot \sigma_{\text{global}} \cdot \sqrt{\frac{\ln N}{n_i}}, \quad C = 2$$

The benefit is **scale independence**: $\sigma_{\text{global}}$ automatically adapts to the overall magnitude of Q-values in the entire search, pooling far more samples than per-node local variance. With $C=2$, the rule tends to generalize better than any single fixed constant across games whose rewards live on different scales.


In [79]:
class MCTSNodeGlobalStd(MCTSNode):
    """MCTSNode using Global Std strategy (Schmocker et al., 2024)."""

    def __init__(self, state: GameState, parent=None, action=None, tree_stats=None):
        super().__init__(state, parent, action)
        self.tree_stats = tree_stats if tree_stats is not None else {"sum_q": 0.0, "sum_q2": 0.0, "count": 0}

    def _global_std(self):
        s = self.tree_stats
        if s["count"] < 2:
            return 0.25
        mean = s["sum_q"] / s["count"]
        var = max(s["sum_q2"] / s["count"] - mean * mean, 0.0)
        return math.sqrt(var) if var > 0 else 0.01

    def selectChild(self, c=2.0):
        for child in self.children:
            if child.visits == 0:
                return child

        sigma_g = self._global_std()
        logN = math.log(self.visits)

        def ucb_gstd(child):
            exploit = child.wins / child.visits
            explore = c * sigma_g * math.sqrt(logN / child.visits)
            return exploit + explore

        return max(self.children, key=ucb_gstd)

    def expand(self):
        (col, t) = self.untriedActions.pop()
        newState = deepcopy(self.state)
        newState.move(col, t)
        child = MCTSNodeGlobalStd(newState, parent=self, action=(col, t),
                                   tree_stats=self.tree_stats)
        self.children.append(child)
        return child

    def backpropagate(self, winner):
        self.visits += 1
        if winner is None:
            reward = 0.5
        elif winner == self.player:
            reward = 1.0
        else:
            reward = 0.0
        self.wins += reward

        if self.visits > 1:
            q = self.wins / self.visits
            s = self.tree_stats
            s["sum_q"] += q
            s["sum_q2"] += q * q
            s["count"] += 1

        if self.parent:
            self.parent.backpropagate(winner)


def mctsGlobalStdSearch(rootState: GameState, iterations=500):
    tree_stats = {"sum_q": 0.0, "sum_q2": 0.0, "count": 0}
    root = MCTSNodeGlobalStd(rootState, tree_stats=tree_stats)
    for _ in range(iterations):
        node = root
        while not node.isTerminal() and node.fullyExpanded():
            node = node.selectChild()
        if not node.isTerminal() and not node.fullyExpanded():
            node = node.expand()
        winner = node.rollout()
        node.backpropagate(winner)
    best = max(root.children, key=lambda c: c.visits)
    return best.action

## Heuristic Rollout MCTS

Rather than purely random rollouts, simulations follow lightweight domain knowledge:

1. **Immediate win**: if a winning move exists, play it  
2. **Centre bias**: central columns are usually more valuable in Connect Four / Pop Out  
3. **Epsilon-greedy**: with probability $\epsilon = 0.15$, play uniformly at random to keep diversity  

This raises rollout quality and yields sharper value estimates at internal nodes.


In [80]:
CENTER_WEIGHTS = {0: 1, 1: 2, 2: 3, 3: 4, 4: 3, 5: 2, 6: 1}

class MCTSNodeHeuristic(MCTSNode):
    """MCTSNode with heuristic-guided rollout policy."""

    def _check_winning_move(self, state, moves):
        for m in moves:
            s = deepcopy(state)
            s.move(m[0], m[1])
            if s.gameOver and s.winner == state.player:
                return m
        return None

    def _heuristic_choice(self, state, moves, epsilon=0.15):
        if random.random() < epsilon:
            return random.choice(moves)

        win = self._check_winning_move(state, moves)
        if win:
            return win

        insert_moves = [m for m in moves if m[1] == 'i']
        if insert_moves:
            weights = [CENTER_WEIGHTS.get(m[0], 1) for m in insert_moves]
            return random.choices(insert_moves, weights=weights, k=1)[0]

        return random.choice(moves)

    def rollout(self):
        state = deepcopy(self.state)
        depth = 0
        while not state.gameOver:
            moves = state.getValid()
            if not moves:
                break
            m = self._heuristic_choice(state, moves)
            state.move(m[0], m[1])
            depth += 1
            if depth > 80:
                break
        return state.winner

    def expand(self):
        (col, t) = self.untriedActions.pop()
        newState = deepcopy(self.state)
        newState.move(col, t)
        child = MCTSNodeHeuristic(newState, parent=self, action=(col, t))
        self.children.append(child)
        return child


def mctsHeuristicSearch(rootState: GameState, iterations=500):
    root = MCTSNodeHeuristic(rootState)
    for _ in range(iterations):
        node = root
        while not node.isTerminal() and node.fullyExpanded():
            node = node.selectChild()
        if not node.isTerminal() and not node.fullyExpanded():
            node = node.expand()
        winner = node.rollout()
        node.backpropagate(winner)
    best = max(root.children, key=lambda c: c.visits)
    return best.action

## RAVE (Rapid Action Value Estimation)

RAVE keeps AMAF (*All-Moves-As-First*) statistics: whenever a move appears anywhere inside a simulation, that simulation's outcome provides evidence for that move from every ancestor position along the path.

The score blends the local MCTS statistics with the global AMAF estimate:

$$V_{RAVE}(n) = (1-\beta) \cdot \frac{Q_{MCTS}}{N_{MCTS}} + \beta \cdot \frac{Q_{AMAF}}{N_{AMAF}} + C\sqrt{\frac{\ln N_{parent}}{N}}$$

with $\beta = \frac{N_{AMAF}}{N_{MCTS} + N_{AMAF} + 4b^2 \cdot N_{MCTS} \cdot N_{AMAF}}$, which shrinks as the per-node MCTS counts become trustworthy.


In [81]:
class MCTSNodeRAVE(MCTSNode):
    """MCTSNode with AMAF statistics for RAVE."""
    def __init__(self, state: GameState, parent=None, action=None):
        super().__init__(state, parent, action)
        self.AMAF_N = 0
        self.AMAF_Q = 0.0

    def selectChild(self, c=1.4, b=0.5):
        for child in self.children:
            if child.visits == 0:
                return child

        def ucb_rave(child):
            q_mcts = child.wins / child.visits
            explore = c * math.sqrt(math.log(self.visits) / child.visits)
            if child.AMAF_N == 0:
                return q_mcts + explore
            q_amaf = child.AMAF_Q / child.AMAF_N
            beta = child.AMAF_N / (child.visits + child.AMAF_N + 4 * b * b * child.visits * child.AMAF_N)
            return (1 - beta) * q_mcts + beta * q_amaf + explore

        return max(self.children, key=ucb_rave)

    def expand(self):
        (col, t) = self.untriedActions.pop()
        newState = deepcopy(self.state)
        newState.move(col, t)
        child = MCTSNodeRAVE(newState, parent=self, action=(col, t))
        self.children.append(child)
        return child

    def rollout(self):
        state = deepcopy(self.state)
        moves_played = []
        while not state.gameOver:
            moves = state.getValid()
            if not moves:
                break
            m = random.choice(moves)
            moves_played.append((m, state.player))
            state.move(m[0], m[1])
        return state.winner, moves_played


def _update_amaf(node, moves_played, outcome):
    """Update AMAF stats for ancestors sharing moves from the rollout."""
    rollout_moves_p1 = {m for m, p in moves_played if p == 1}
    rollout_moves_p2 = {m for m, p in moves_played if p == 2}

    cur = node
    while cur is not None and cur.parent is not None:
        parent = cur.parent
        player_at_parent = cur.player
        relevant = rollout_moves_p1 if player_at_parent == 1 else rollout_moves_p2
        for sibling in parent.children:
            if sibling.action in relevant:
                sibling.AMAF_N += 1
                if outcome is None:
                    rew = 0.5
                elif outcome == player_at_parent:
                    rew = 1.0
                else:
                    rew = 0.0
                sibling.AMAF_Q += rew
        cur = cur.parent


def mctsRAVESearch(rootState: GameState, iterations=500):
    root = MCTSNodeRAVE(rootState)
    for _ in range(iterations):
        node = root
        while not node.isTerminal() and node.fullyExpanded():
            node = node.selectChild()
        if not node.isTerminal() and not node.fullyExpanded():
            node = node.expand()

        result = node.rollout()
        winner, moves_played = result

        node.backpropagate(winner)
        _update_amaf(node, moves_played, winner)

    best = max(root.children, key=lambda c: c.visits)
    return best.action

## Progressive Widening + Early Termination

**Progressive widening:** instead of expanding every legal child immediately, we allow at most $k \cdot N^\alpha$ children, where $N$ is the parent's visit count and $\alpha \approx 0.5$. Promising moves are explored first; weaker moves enter only after additional visits accumulate.

**Early termination:** if a rollout exceeds a move budget without reaching terminal states, we score the leaf with a cheap heuristic (material, centre occupancy, etc.) instead of rolling forever.

Together these knobs implement the brief's request to *vary how many children each node expands*.

In [82]:
class MCTSNodeProgressive(MCTSNode):
    """MCTSNode with progressive widening and early termination."""

    def __init__(self, state: GameState, parent=None, action=None, k=2.0, alpha=0.5, max_depth=60):
        super().__init__(state, parent, action)
        self.k = k
        self.alpha = alpha
        self.max_depth = max_depth

    def _max_children(self):
        return max(1, int(self.k * (max(self.visits, 1) ** self.alpha)))

    def expand(self):
        (col, t) = self.untriedActions.pop()
        newState = deepcopy(self.state)
        newState.move(col, t)
        child = MCTSNodeProgressive(newState, parent=self, action=(col, t),
                                     k=self.k, alpha=self.alpha, max_depth=self.max_depth)
        self.children.append(child)
        return child

    def _heuristic_eval(self, state):
        """Material + centrality heuristic for early termination."""
        p1_score, p2_score = 0.0, 0.0
        for col in range(state.w):
            for row in range(len(state.board[col])):
                piece = state.board[col][row]
                center_bonus = 1.0 + (3 - abs(col - 3)) * 0.1
                if piece == 1:
                    p1_score += center_bonus
                elif piece == 2:
                    p2_score += center_bonus
        if p1_score > p2_score:
            return 1
        elif p2_score > p1_score:
            return 2
        return None

    def rollout(self):
        state = deepcopy(self.state)
        depth = 0
        while not state.gameOver:
            moves = state.getValid()
            if not moves:
                break
            m = random.choice(moves)
            state.move(m[0], m[1])
            depth += 1
            if depth >= self.max_depth:
                return self._heuristic_eval(state)
        return state.winner


def mctsProgressiveSearch(rootState: GameState, iterations=500, k=2.0, alpha=0.5, max_depth=60):
    root = MCTSNodeProgressive(rootState, k=k, alpha=alpha, max_depth=max_depth)
    for _ in range(iterations):
        node = root

        while not node.isTerminal() and node.fullyExpanded():
            allowed = node._max_children()
            if len(node.children) < allowed and len(node.untriedActions) > 0:
                break
            node = node.selectChild()

        if not node.isTerminal() and len(node.untriedActions) > 0:
            node = node.expand()

        winner = node.rollout()
        node.backpropagate(winner)

    best = max(root.children, key=lambda c: c.visits)
    return best.action

## Forking (Scenario Diversification)

The `forkProb` hyper-parameter (default 4%) switches on *forking*. Roughly 4% of the time the driver spawns a parallel branch (*side position*) where the agent must play something **different** from its usual best reply.

**Sampling the alternative move** (the nominal best move is forbidden):

- **70%:** Softmax over visit counts with temperature $T = 1.0$
- **25%:** High temperature ($T = 2.0$)—use $P \propto N^{1/T}$ to flatten mass onto rarely visited moves
- **5%:** Uniform random legal move

**Why bother:** we accumulate counterfactual data—“what would have happened if I had played *there*?”—which teaches the agent to refute blunders that vanilla search would never revisit.

In [83]:
def _choose_forking_move(children, best_action, legal_moves, temp=1.0):
    """Choose an alternative move for forking, excluding the best action."""
    alt_children = [c for c in children if c.action != best_action and c.visits > 0]
    alt_legal = [m for m in legal_moves if m != best_action]

    if not alt_legal:
        return None

    roll = random.random()

    if roll < 0.70 and alt_children:
        visits = np.array([c.visits for c in alt_children], dtype=float)
        probs = visits / visits.sum()
        idx = np.random.choice(len(alt_children), p=probs)
        return alt_children[idx].action

    elif roll < 0.95 and alt_children:
        visits = np.array([c.visits for c in alt_children], dtype=float)
        adjusted = np.power(visits, 1.0 / 2.0)
        probs = adjusted / adjusted.sum()
        idx = np.random.choice(len(alt_children), p=probs)
        return alt_children[idx].action

    else:
        return random.choice(alt_legal)


def mctsSearchWithForking(rootState: GameState, iterations=500, forkProb=0.04):
    """MCTS search that occasionally returns a forked alternative move.
    Returns (best_action, fork_action_or_None)."""
    root = MCTSNode(rootState)
    for _ in range(iterations):
        node = root
        while not node.isTerminal() and node.fullyExpanded():
            node = node.selectChild()
        if not node.isTerminal() and not node.fullyExpanded():
            node = node.expand()
        winner = node.rollout()
        node.backpropagate(winner)

    best = max(root.children, key=lambda c: c.visits)
    best_action = best.action

    fork_action = None
    if random.random() < forkProb:
        legal = rootState.getValid()
        fork_action = _choose_forking_move(root.children, best_action, legal)

    return best_action, fork_action

## Root Policy Softmax Temperature

To avoid premature convergence at the search root, instead of picking the most visited move deterministically ($\arg\max N$), we sample from a temperature-scaled distribution over visit counts:

$$P(a) = \frac{N(a)^{1/T}}{\sum_{a'} N(a')^{1/T}}$$

- $T > 1$: flattens the distribution—covers a wider palette of moves
- $T = 1$: proportional to raw visit counts
- $T < 1$: sharpens the distribution—more greedy
- $T \to 0$: converges to $\arg\max$

We initialise temperature at $1.2$ and multiply it by $0.6$ after every move until it hits a floor of $0.01$.

**Motivation** (SAI / Leela Zero): with $T = 1.0$, the policy often collapses too quickly, starving alternative lines of further simulations. Higher early temperatures preserve uncertainty and yield richer supervision signals.

In [84]:
def mctsSearchWithTemperature(rootState: GameState, iterations=500, temp=1.2):
    """MCTS search using softmax temperature for root move selection."""
    root = MCTSNode(rootState)
    for _ in range(iterations):
        node = root
        while not node.isTerminal() and node.fullyExpanded():
            node = node.selectChild()
        if not node.isTerminal() and not node.fullyExpanded():
            node = node.expand()
        winner = node.rollout()
        node.backpropagate(winner)

    if not root.children:
        return None

    if temp <= 0.01:
        best = max(root.children, key=lambda c: c.visits)
        return best.action

    visits = np.array([c.visits for c in root.children], dtype=float)
    adjusted = np.power(visits, 1.0 / temp)
    probs = adjusted / adjusted.sum()
    idx = np.random.choice(len(root.children), p=probs)
    return root.children[idx].action


class TemperatureSchedule:
    """Manages temperature decay across moves in a game."""
    def __init__(self, initial=1.2, decay=0.6, minimum=0.01):
        self.temp = initial
        self.decay = decay
        self.minimum = minimum

    def get(self):
        return self.temp

    def step(self):
        self.temp = max(self.minimum, self.temp * self.decay)

## ELO Rating System & Tournament

We track Elo ratings for every MCTS variant. After each finished game we update both competitors and record how standings drift across tournament rounds.

**Elo update:**

$$E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}, \quad R'_A = R_A + K(S_A - E_A)$$

- $K = 32$ (learning rate)
- Starting rating: $1500$ for all agents
- Round-robin schedule: every pairing plays twice with alternating first move

In [65]:
def expected_score(ra, rb):
    return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))


def update_elo(ra, rb, sa, K=32):
    """Update ELO ratings given actual score sa for player A (1=win, 0.5=draw, 0=loss)."""
    ea = expected_score(ra, rb)
    eb = 1.0 - ea
    sb = 1.0 - sa
    return ra + K * (sa - ea), rb + K * (sb - eb)


def play_one_game(search_fn_1, search_fn_2, iters=500):
    """Play a single game. search_fn_i(state, iterations) -> (col, type).
    Returns 1 if player 1 wins, 2 if player 2 wins, 0 for draw."""
    g = GameState()
    while not g.gameOver:
        if g.player == 1:
            action = search_fn_1(g, iterations=iters)
        else:
            action = search_fn_2(g, iterations=iters)
        if action is None:
            break
        g.move(action[0], action[1])
    return g.winner if g.winner else 0


def mctsForkingPlay(rootState: GameState, iterations=500):
    """Wrapper: plays the fork move when available, otherwise the best move."""
    best, fork = mctsSearchWithForking(rootState, iterations=iterations, forkProb=0.04)
    return fork if fork is not None else best


def mctsTemperaturePlay(rootState: GameState, iterations=500,
                        initial=1.2, decay=0.6, minimum=0.01):
    """Temperature player with decay based on move number.
    Uses piece count on the board as a proxy for how far into the game we are."""
    move_number = sum(len(col) for col in rootState.board)
    temp = max(minimum, initial * (decay ** move_number))
    return mctsSearchWithTemperature(rootState, iterations=iterations, temp=temp)


ALGORITHMS = {
    "Baseline":     mctsSearch,
    "Stupid":       mctstupidSearch,
    "Dynamic":      mctsDynamicSearch,
    "UCB-V":        mctsUCBVSearch,
    "GlobalStd":    mctsGlobalStdSearch,
    "Heuristic":    mctsHeuristicSearch,
    "RAVE":         mctsRAVESearch,
    "Progressive":  mctsProgressiveSearch,
    "Forking":      mctsForkingPlay,
    "Temperature":  mctsTemperaturePlay,
}


def run_elo_tournament(algorithms=None, games_per_pair=10, iters=500, checkpoints=None):
    """Round-robin ELO tournament with tracking.
    
    Args:
        algorithms: dict of {name: search_fn}
        games_per_pair: total games each pair plays
        iters: MCTS iterations per move
        checkpoints: list of game counts at which to snapshot ELO (auto if None)
    
    Returns:
        elo_ratings: final dict {name: elo}
        elo_history: list of (game_number, {name: elo}) snapshots
    """
    if algorithms is None:
        algorithms = ALGORITHMS

    names = list(algorithms.keys())
    fns = {n: algorithms[n] for n in names}
    elo = {n: 1500.0 for n in names}
    elo_history = []
    total_games_played = 0

    pairs = [(names[i], names[j]) for i in range(len(names)) for j in range(i+1, len(names))]
    total_games = len(pairs) * games_per_pair

    if checkpoints is None:
        step = max(1, total_games // 10)
        checkpoints = set(range(step, total_games + 1, step))
        checkpoints.add(total_games)
    else:
        checkpoints = set(checkpoints)

    elo_history.append((0, dict(elo)))

    schedule = []
    for game_i in range(games_per_pair):
        for n1, n2 in pairs:
            if game_i % 2 == 0:
                schedule.append((n1, n2))
            else:
                schedule.append((n2, n1))

    for p1_name, p2_name in schedule:
        result = play_one_game(fns[p1_name], fns[p2_name], iters=iters)

        if result == 1:
            sa = 1.0
        elif result == 2:
            sa = 0.0
        else:
            sa = 0.5

        elo[p1_name], elo[p2_name] = update_elo(elo[p1_name], elo[p2_name], sa)
        total_games_played += 1

        if total_games_played in checkpoints:
            elo_history.append((total_games_played, dict(elo)))
            ranked = sorted(elo.items(), key=lambda x: -x[1])
            print(f"[Game {total_games_played}/{total_games}] " +
                  " | ".join(f"{n}: {int(r)}" for n, r in ranked))

    return elo, elo_history


def elo_history_to_dataframe(elo_history):
    """pandas DataFrame for easy visualization."""
    rows = []
    for game_num, ratings in elo_history:
        row = {"Game": game_num}
        row.update(ratings)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("Game")
    return df.round(0).astype(int)

### Run Elo tournament
Round-robin among every MCTS flavour in the notebook. The dataframe plots how each rating evolves as games stream in.

In [68]:
print("=== ELO TOURNAMENT — MCTS VARIANTS ===\n")

elo_final, elo_hist = run_elo_tournament(
    algorithms=ALGORITHMS,
    games_per_pair=10,
    iters=500,
)

print("\n=== ELO FINAL ===")
for name, rating in sorted(elo_final.items(), key=lambda x: -x[1]):
    print(f"  {name:20s} {int(rating)}")

print("\n=== EVOLUÇÃO DO ELO ===")
df_elo = elo_history_to_dataframe(elo_hist)
print(df_elo.to_string())

=== TORNEIO ELO — VARIANTES MCTS ===



KeyboardInterrupt: 

# DECISION TREE

## DT Model
The algorithm splits on discrete values rather than continuous and converts floats to int when classifying.

In [85]:
def entropy(col):
    elements, counts = np.unique(col, return_counts=True)
    entropyValue = -np.sum([
        (counts[i] / np.sum(counts)) * np.log2(counts[i] / np.sum(counts))
        for i in range(len(elements))
    ])
    return entropyValue

def informationGain(data, feature, target="class"):
    total_entropy = entropy(data[target])
    values, counts = np.unique(data[feature], return_counts=True)

    weighted_entropy = np.sum([
        (counts[i] / np.sum(counts)) *
        entropy(data[data[feature] == values[i]][target])
        for i in range(len(values))
    ])

    return total_entropy - weighted_entropy

def id3(data, ogData, features, target="class", parentNodeClass=None):
    """Recursive ID3 Construction"""
    # Stopping Conditions
    if len(np.unique(data[target])) == 1:
        return np.unique(data[target])[0]

    if len(data) == 0:
        return np.unique(ogData[target])[np.argmax(
            np.unique(ogData[target], return_counts=True)[1])]

    if len(features) == 0:
        return parentNodeClass

    # Split dataset on best feature
    parentNodeClass = np.unique(data[target])[np.argmax(
        np.unique(data[target], return_counts=True)[1])]
    
    gains = [informationGain(data, feature, target) for feature in features]
    bestFeature = features[np.argmax(gains)]

    tree = {bestFeature: {}}
    features = [f for f in features if f != bestFeature]

    # Recursive subtree construction
    for value in np.unique(data[bestFeature]):
        subset = data[data[bestFeature] == value]
        subtree = id3(subset, ogData, features, target, parentNodeClass)
        tree[bestFeature][value] = subtree

    return tree

# -- Straight Slop:

def classify(tree, row):
    def getAll(node):
        if not isinstance(node, dict):
            return [node]
        out=[]
        for child in node.values():
            out.extend(getAll(child))
        return out

    if not isinstance(tree, dict):
        return tree
    feature = next(iter(tree))
    branch = tree[feature]
    
    for key in branch.keys():
        # threshold = float(key) # TODO: find bbetter name

        if row[feature] == key:
            return classify(branch[key], row)
    
    # In case no condition matched
    leaves=getAll(branch)
    if not leaves:
        return None
    uniq,counts = np.unique(np.asarray(leaves,dtype=object), return_counts=True)
    return uniq[np.argmax(counts)]

# -- Better visualization of the tree
def printT(tree, indent=0):
    pad = "  " * indent
    if not isinstance(tree, dict):
        print(f"{pad}-> {tree}")
        return
    for feature, branches in tree.items():
        print(f"{pad}{feature}")
        for value, subtree in branches.items():
            print(f"{pad}  [{value}]")
            printT(subtree, indent + 2)

### Testing DT with Iris

#### Creating the Tree

In [86]:
data = pd.read_csv("iris.csv")
features = list(data.columns[1:-1])
tree = id3(data,data,features, target="class")
printT(tree)

petallength
  [1.0]
    -> Iris-setosa
  [1.1]
    -> Iris-setosa
  [1.2]
    -> Iris-setosa
  [1.3]
    -> Iris-setosa
  [1.4]
    -> Iris-setosa
  [1.5]
    -> Iris-setosa
  [1.6]
    -> Iris-setosa
  [1.7]
    -> Iris-setosa
  [1.9]
    -> Iris-setosa
  [3.0]
    -> Iris-versicolor
  [3.3]
    -> Iris-versicolor
  [3.5]
    -> Iris-versicolor
  [3.6]
    -> Iris-versicolor
  [3.7]
    -> Iris-versicolor
  [3.8]
    -> Iris-versicolor
  [3.9]
    -> Iris-versicolor
  [4.0]
    -> Iris-versicolor
  [4.1]
    -> Iris-versicolor
  [4.2]
    -> Iris-versicolor
  [4.3]
    -> Iris-versicolor
  [4.4]
    -> Iris-versicolor
  [4.5]
    sepallength
      [4.9]
        -> Iris-virginica
      [5.4]
        -> Iris-versicolor
      [5.6]
        -> Iris-versicolor
      [5.7]
        -> Iris-versicolor
      [6.0]
        -> Iris-versicolor
      [6.2]
        -> Iris-versicolor
      [6.4]
        -> Iris-versicolor
  [4.6]
    -> Iris-versicolor
  [4.7]
    -> Iris-versicolor
  [4.8]
    sep

#### Testing the tree

In [87]:
shufData = data.sample(frac=1).reset_index(drop=True)
splitIx = int(0.75*len(data))
trainData = shufData.iloc[:splitIx]
testData = shufData.iloc[splitIx:]

tree2 = id3(trainData,trainData,features, target="class")

count=0
for _, row in testData.iterrows():
    if classify(tree2, row) == row['class']:
        count += 1
print(count / len(testData))

0.9210526315789473


## Creating Pop Out Dataset

**Teacher agents (top two Elo scores)** always open as player 1—**Heuristic** and **GlobalStd**. Every row logs whichever move they execute.

**Player-two rotation:** Heuristic → Temperature → Baseline → Progressive → Forking → RAVE → Dynamic → Stupid → **GlobalStd**. GlobalStd is duplicated at the end so we observe symmetric Heuristic ↔ GlobalStd fixtures; otherwise GlobalStd would only appear while sitting on player 1 against Heuristic.

**Logging player 2:** enabled solely when the opponent is another strong teacher (**Heuristic** or **GlobalStd**). Lightweight opponents contribute supervision only through the primary teacher's labels.

`logMove` still deduplicates identical `(board snapshot, move)` pairs before appending to the CSV.

In [88]:
# -- Formatting --

def flatBoard(board):
    """Makes the 2-dimentional board into a 1-dimentional list"""
    out = []
    for i in range (6):
        for j in range (7):
            if j < len(board[i]):
                out.append(board[i][j])
            else:
                out.append(0)
    return out

def moveToInt(c,t):
    i = c<<1
    if t == 'p':
        i+=1
    return i

def intToMove(i):
    c = i>>1
    if i&1 == 0:
        t='i'
    else:
        t='p'
    return (c,t)

# -- Creating the dataset --
file="con4pOut.csv"
curDataSet=set()

try:
    with open(file, 'r') as f:
        fRead = csv.reader(f)
        for row in fRead:
            if len(row) != 43:
                continue
            if row[-1].strip().lower() == "move":
                continue  # header row
            # Same canonical key as logMove (all str), for dedup across runs
            k = tuple(str(int(float(cell))) for cell in row)
            curDataSet.add(k)

    fp = open(file,'a',newline='')
    fWrite=csv.writer(fp)

except FileNotFoundError:
    fp = open(file,'a',newline='')
    fWrite=csv.writer(fp)

    fWrite.writerow([str(n) for n in range(42)] + ["move"])
    print(f"Created {file}")

def logMove(board,move):
    fB = flatBoard(board)
    k=tuple(fB + [move])
    if k in curDataSet:
        return
    curDataSet.add(k)
    fWrite.writerow(fB+[move])

# -- Playing the game --
PRINCIPALS = {
    "Heuristic": mctsHeuristicSearch,
    "GlobalStd": mctsGlobalStdSearch,
}
STRONG_VARIANTS = frozenset({"Heuristic", "GlobalStd"})

ROTATION = [
    ("Heuristic", mctsHeuristicSearch),
    ("Temperature", mctsTemperaturePlay),
    ("Baseline", mctsSearch),
    ("Progressive", mctsProgressiveSearch),
    ("Forking", mctsForkingPlay),
    ("RAVE", mctsRAVESearch),
    ("Dynamic", mctsDynamicSearch),
    ("Stupid", mctstupidSearch),
    ("GlobalStd", mctsGlobalStdSearch),
]

MATCHUPS = []
for p1_name, p1_fn in PRINCIPALS.items():
    for p2_name, p2_fn in ROTATION:
        log_p2 = p2_name in STRONG_VARIANTS
        MATCHUPS.append((p1_name, p1_fn, p2_name, p2_fn, True, log_p2))

GAMES_PER_MATCHUP = 10
ITERS = 1000

for p1_name, p1_fn, p2_name, p2_fn, log_p1, log_p2 in MATCHUPS:
    for game_i in range(GAMES_PER_MATCHUP):
        g = GameState()
        while not g.gameOver:
            if g.player == 1:
                (c, t) = p1_fn(g, iterations=ITERS)
                if log_p1:
                    logMove(g.board, moveToInt(c, t))
                g.move(c, t)
            else:
                (c, t) = p2_fn(g, iterations=ITERS)
                if log_p2:
                    logMove(g.board, moveToInt(c, t))
                g.move(c, t)
        print(f"[{p1_name} vs {p2_name}] Game {game_i+1} — Winner: {g.winner}")

# -- Close file --
fp.close()

[Heuristic vs Heuristic] Game 1 — Winner: 1
[Heuristic vs Heuristic] Game 2 — Winner: 1
[Heuristic vs Heuristic] Game 3 — Winner: 1
[Heuristic vs Heuristic] Game 4 — Winner: 2
[Heuristic vs Heuristic] Game 5 — Winner: 1
[Heuristic vs Heuristic] Game 6 — Winner: 1
[Heuristic vs Heuristic] Game 7 — Winner: 1
[Heuristic vs Heuristic] Game 8 — Winner: 2
[Heuristic vs Heuristic] Game 9 — Winner: 1
[Heuristic vs Heuristic] Game 10 — Winner: 1
[Heuristic vs Temperature] Game 1 — Winner: 1
[Heuristic vs Temperature] Game 2 — Winner: 1
[Heuristic vs Temperature] Game 3 — Winner: 1
[Heuristic vs Temperature] Game 4 — Winner: 1
[Heuristic vs Temperature] Game 5 — Winner: 1
[Heuristic vs Temperature] Game 6 — Winner: 2
[Heuristic vs Temperature] Game 7 — Winner: 1
[Heuristic vs Temperature] Game 8 — Winner: 1
[Heuristic vs Temperature] Game 9 — Winner: 1
[Heuristic vs Temperature] Game 10 — Winner: 1
[Heuristic vs Baseline] Game 1 — Winner: 1
[Heuristic vs Baseline] Game 2 — Winner: 1
[Heuristic v

In [ ]:
# Heuristic vs Stupid-only dataset → `con4pOutn.csv`
# Log Heuristic moves only (Stupid is never recorded).

def flatBoard_hs(board):
    out = []
    for i in range(6):
        for j in range(7):
            if j < len(board[i]):
                out.append(board[i][j])
            else:
                out.append(0)
    return out


def move_to_int_hs(c, t):
    i = c << 1
    if t == "p":
        i += 1
    return i


file_n = "con4pOutn.csv"
cur_n = set()

try:
    with open(file_n, "r") as f:
        for row in csv.reader(f):
            if len(row) != 43:
                continue
            if row[-1].strip().lower() == "move":
                continue
            cur_n.add(tuple(str(int(float(cell))) for cell in row))
    fp_n = open(file_n, "a", newline="")
    w_n = csv.writer(fp_n)
except FileNotFoundError:
    fp_n = open(file_n, "a", newline="")
    w_n = csv.writer(fp_n)
    w_n.writerow([str(n) for n in range(42)] + ["move"])
    print(f"Created {file_n}")


def log_heuristic(board, move_int):
    fb = flatBoard_hs(board)
    key = tuple(fb + [move_int])
    if key in cur_n:
        return
    cur_n.add(key)
    w_n.writerow(fb + [move_int])


ITERS_HS = 1000
GAMES_HEUR_VS_STUPID = 200

for game_i in range(GAMES_HEUR_VS_STUPID):
    g = GameState()
    heuristic_is_p1 = game_i % 2 == 0
    while not g.gameOver:
        if g.player == 1:
            if heuristic_is_p1:
                c, t = mctsHeuristicSearch(g, iterations=ITERS_HS)
                log_heuristic(g.board, move_to_int_hs(c, t))
                g.move(c, t)
            else:
                c, t = mctstupidSearch(g, iterations=ITERS_HS)
                g.move(c, t)
        else:
            if heuristic_is_p1:
                c, t = mctstupidSearch(g, iterations=ITERS_HS)
                g.move(c, t)
            else:
                c, t = mctsHeuristicSearch(g, iterations=ITERS_HS)
                log_heuristic(g.board, move_to_int_hs(c, t))
                g.move(c, t)
    who = "Heuristic=P1" if heuristic_is_p1 else "Heuristic=P2"
    print(f"[Heuristic vs Stupid | {who}] Game {game_i + 1} — Winner: {g.winner}")

fp_n.close()

Created con4pOutn.csv
[Heuristic vs Stupid | Heuristic=P1] Game 1 — Winner: 1
[Heuristic vs Stupid | Heuristic=P2] Game 2 — Winner: 2
[Heuristic vs Stupid | Heuristic=P1] Game 3 — Winner: 1
[Heuristic vs Stupid | Heuristic=P2] Game 4 — Winner: 2
[Heuristic vs Stupid | Heuristic=P1] Game 5 — Winner: 1
[Heuristic vs Stupid | Heuristic=P2] Game 6 — Winner: 2
[Heuristic vs Stupid | Heuristic=P1] Game 7 — Winner: 1
[Heuristic vs Stupid | Heuristic=P2] Game 8 — Winner: 2
[Heuristic vs Stupid | Heuristic=P1] Game 9 — Winner: 1
[Heuristic vs Stupid | Heuristic=P2] Game 10 — Winner: 1


### Making the tree

Primary decision tree trained on `con4pOut.csv` (`popOutTree`). Companion **stupid-policy** tree trained on `con4pOutn.csv` (`popOutTreeStupid`).

In [97]:
data_main = pd.read_csv("con4pOut.csv")
features = list(data_main.columns[:-1])
popOutTree = id3(data_main, data_main, features, target="move")

data_stupid = pd.read_csv("con4pOutn.csv")
features_s = list(data_stupid.columns[:-1])
popOutTreeStupid = id3(data_stupid, data_stupid, features_s, target="move")

### Testing DT for Pop Out

In [98]:
def dt_play_with_tree(g, tree, feature_columns):
    fb = flatBoard(g.board)
    row = {feature_columns[i]: int(fb[i]) for i in range(len(feature_columns))}
    move = classify(tree, row)
    moves = g.getValid()
    if not moves:
        return (0, "i")
    if move is None:
        return random.choice(moves)
    c, t = intToMove(int(move))
    if (c, t) not in moves:
        return random.choice(moves)
    return (c, t)


def dtPlay(g):
    return dt_play_with_tree(g, popOutTree, features)


def dtPlayStupid(g):
    return dt_play_with_tree(g, popOutTreeStupid, features_s)


#### Human vs DT

In [94]:
g=GameState()

g.printB()
while not g.gameOver:
    if g.player == 1:
        t=input("Type: ") # 'i' <- insert | 'p' <- pop
        c=int(input("Column: "))

        if not g.move(c,t):
            print("Invalid Move")
        else:
            clear_output()
            g.printB()
    else:
        (c,t)=dtPlay(g)

        g.move(c,t)
        clear_output()
        g.printB()

print("--- Winner:",g.winner,"---")

+-------+
|       |
|       |
|       |
|     X |
|   O XX|
| OOOOXX|
+-------+
|0123456|
+-------+
--- Winner: 2 ---


#### DT vs MCTS

In [95]:
for _ in range (10):
    g=GameState()

    #g.printB()
    while not g.gameOver:
        if g.player == 2:
            (c,t) = mctstupidSearch(g,iterations=1000)
            g.move(c,t)
        else:
            (c,t) = dtPlay(g)
            g.move(c,t)

    print("--- Winner:",g.winner,"---")

--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 2 ---
--- Winner: 1 ---
--- Winner: 2 ---
--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---
--- Winner: 1 ---
